# Farm360 Crop Disease V2 Model Training (Google Colab / Kaggle)

This notebook trains the **EfficientNetV2-S** crop disease classification model on 42 categories using a GPU runtime.

### Prerequisites:
1. Set Colab Runtime to **GPU** (`Runtime -> Change runtime type -> T4 GPU`)
2. Upload the `clean/` dataset folder as `clean.zip` and unzip it in Colab.

In [ ]:
# 1. Unzip dataset
!unzip -q clean.zip -d clean_dataset/

In [ ]:
# 2. Clone or copy training script
with open("train_gpu.py", "w", encoding="utf-8") as f:
    f.write('''
import os
import sys
import time
import json
import random
import argparse
from datetime import datetime
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import precision_recall_fscore_support

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

def create_experiment_dir(base_dir):
    os.makedirs(base_dir, exist_ok=True)
    runs = [d for d in os.listdir(base_dir) if d.startswith("run_") and os.path.isdir(os.path.join(base_dir, d))]
    next_run_num = max([int(r.split("_")[1]) for r in runs]) + 1 if runs else 1
    run_dir = os.path.join(base_dir, f"run_{next_run_num:03d}")
    os.makedirs(run_dir, exist_ok=True)
    return run_dir, f"run_{next_run_num:03d}"

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=35)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--dataset-dir", type=str, default="")
    args = parser.parse_args()

    print("🚀 Starting GPU-Accelerated Crop Disease V2 Training")
    set_seed(42)
    
    train_dir = os.path.join(args.dataset_dir, "train")
    val_dir = os.path.join(args.dataset_dir, "validation")
    experiments_base = "experiments"

    run_dir, run_name = create_experiment_dir(experiments_base)
    print(f"✓ Created experiment directory: {run_dir}")

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = ImageFolder(train_dir, transform=train_transform, allow_empty=True)
    val_dataset = ImageFolder(val_dir, transform=val_transform, allow_empty=True)
    classes = train_dataset.classes

    class_counts = {}
    for _, label in train_dataset.samples:
        class_counts[label] = class_counts.get(label, 0) + 1
    class_weights = [1.0 / class_counts[i] if i in class_counts else 1.0 for i in range(len(classes))]
    sample_weights = [class_weights[label] for _, label in train_dataset.samples]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, sampler=sampler, num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False, num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)

    model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, len(classes))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    print(f"✓ Allocated model to training device: {device}")

    criterion = FocalLoss(gamma=2)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10)
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

    best_f1 = 0.0
    best_epoch = 1
    
    for epoch in range(args.epochs):
        model.train()
        running_loss = 0.0
        t0 = time.time()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            
        avg_train_loss = running_loss / len(train_loader)
        scheduler.step()
        epoch_time = time.time() - t0
        
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                    outputs = model(images)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.numpy())
                
        precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
        top1_correct = sum([1 for p, l in zip(all_preds, all_labels) if p == l])
        val_acc = (top1_correct / len(all_labels)) * 100 if all_labels else 0.0
        print(f"Epoch [{epoch+1}/{args.epochs}] ({epoch_time:.1f}s) — Loss: {avg_train_loss:.4f} — Val Acc: {val_acc:.2f}% — Val F1: {f1:.4f}")
        
        if f1 > best_f1:
            best_f1 = f1
            best_epoch = epoch + 1
            torch.save(model.state_dict(), os.path.join(run_dir, "weights.pth"))
            
    print(f"\n🎉 Training complete. Best Epoch: {best_epoch} with Val F1: {best_f1:.4f}")
    
if __name__ == '__main__':
    main()
''')
print("✓ train_gpu.py written successfully.")

In [ ]:
# 3. Execute training
!python train_gpu.py --epochs 35 --batch-size 64 --dataset-dir clean_dataset/clean/

In [ ]:
# 4. Zip results for download
!zip -r experiments.zip experiments/